# 실험 10 | 최종 암종 분류 모델 학습 및 제출 파일 생성

**확정 모델:** `SGDClassifier(loss="log_loss", alpha=0.001, class_weight="balanced")` + **Interaction 특징** (실험 09의 Train 내부 최종 후보 선택 결과 사용)

**데이터 누출 방지**
- Test 데이터는 최종 모델의 `predict`에만 사용합니다. 어떠한 모델/인코더/스케일러/구간 경계도 Test에서 학습하지 않습니다.
- 모든 변이 유전자 열은 Train의 열 이름 및 순서로 고정합니다. `WT`=0, 관측된 비-`WT`=1, 결측=0으로 처리합니다. **결측=0은 고정된 기술적 규칙이며, 생물학적으로 WT를 의미하지 않습니다.**
- 변이 수 구간 경계(q25/q75)와 스케일러는 **전체 Train에서만 fit**합니다.
- 모델 후보는 실험 09의 교차검증으로 이미 결정했습니다. **Test 예측/분포를 보고 모델을 바꾸지 않습니다.**
- 실험 09의 Macro F1은 Train 기반 탐색적 검증 결과이며, 제출 파일에 대한 공식 점수를 뜻하지 않습니다.

**필요 파일:** `data/train(1).csv`, `data/test(2).csv`, `data/sample_submission(1).csv` (파일명은 `train.csv`, `test.csv`, `sample_submission.csv`도 허용). `outputs/X_train_binary.npz`, `outputs/y_train.npy`, `outputs/gene_columns.json`이 있으면 Train 재전처리 대신 해당 결과를 검증 후 사용할 수 있습니다. 순수하게 처음부터 실행할 수도 있습니다.

In [1]:
# 필요 시 터미널: python -m pip install numpy pandas scipy scikit-learn joblib nbformat
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
import joblib

SEED = 42
ROOT = Path.cwd()
DATA = ROOT / 'data'
if not DATA.exists(): DATA = ROOT
OUT = ROOT / 'experiment_10'
OUT.mkdir(parents=True, exist_ok=True)

def locate(names, folders):
    for folder in folders:
        for name in names:
            f = folder / name
            if f.is_file(): return f
    raise FileNotFoundError(f'파일을 찾을 수 없음: {names}, 탐색 폴더: {folders}')

train_path = locate(['train(1).csv','train.csv'], [DATA,ROOT])
test_path = locate(['test(2).csv','test.csv','test(1).csv'], [DATA,ROOT])
sample_path = locate(['sample_submission(1).csv','sample_submission.csv'], [DATA,ROOT])
print('Train:',train_path,'\nTest:',test_path,'\nSample:',sample_path)

Train: /mnt/data/experiment_10_final/data/train(1).csv 
Test: /mnt/data/experiment_10_final/data/test(2).csv 
Sample: /mnt/data/experiment_10_final/data/sample_submission(1).csv


## 1. Train으로 특징 명세 확정

CSV를 분할해서 읽어 메모리 부담을 줄입니다. 학습용 특징 이름과 순서는 Train에서 정하고, Test는 이 순서로만 변환합니다.

In [2]:
# 헤더만 읽어 Train의 feature schema 결정
head = pd.read_csv(train_path, nrows=0)
assert {'ID','SUBCLASS'}.issubset(head.columns)
gene_columns = [c for c in head.columns if c not in ('ID','SUBCLASS')]
assert len(gene_columns)==4384 and len(set(gene_columns))==len(gene_columns)
assert 'SUBCLASS' not in gene_columns and 'ID' not in gene_columns

# 입력 변환: 학습된 통계가 전혀 없는, 고정된 규칙
# WT=0, non-WT=1, NaN=0 (결측은 미관측임을 별도 기록)
def as_binary(frame, genes):
    arr = frame[genes].ne('WT').fillna(False).to_numpy(dtype=np.uint8)
    return sparse.csr_matrix(arr)

xs, ys, train_ids = [], [], []
for chunk in pd.read_csv(train_path, dtype='string', chunksize=300):
    assert all(g in chunk.columns for g in gene_columns)
    assert chunk['SUBCLASS'].notna().all() and chunk['ID'].notna().all()
    xs.append(as_binary(chunk, gene_columns))
    ys.extend(chunk['SUBCLASS'].astype(str).tolist())
    train_ids.extend(chunk['ID'].astype(str).tolist())
X_train = sparse.vstack(xs,format='csr').astype(np.float32)
y_text = np.asarray(ys)
train_ids = np.asarray(train_ids)
assert X_train.shape==(6201,4384)
assert len(np.unique(train_ids))==len(train_ids)
enc=LabelEncoder().fit(y_text)  # Train 라벨에서만 fit
classes=enc.classes_
y=enc.transform(y_text)
assert len(classes)==26
print('Train shape:',X_train.shape,'Classes:',len(classes),'NonWT:',X_train.nnz)

Train shape: (6201, 4384) Classes: 26 NonWT: 218893


## 2. Interaction 특징과 전처리 기준은 Train에서만 학습

In [3]:
# 실험 09의 make_features(..)와 동일한 설계
train_counts = np.diff(X_train.indptr).astype(np.float32)
count_scaler = StandardScaler()
train_scaled = count_scaler.fit_transform(np.log1p(train_counts).reshape(-1,1)).astype(np.float32)
q25, q75 = np.percentile(train_counts,[25,75])

def interaction_transform(X, scaled_count, counts, q25, q75):
    low = (counts<=q25).astype(np.float32)
    high = (counts>=q75).astype(np.float32)
    parts=[X,sparse.csr_matrix(scaled_count),
           X.multiply(low[:,None]).tocsr(),
           X.multiply(high[:,None]).tocsr()]
    return sparse.hstack(parts,format='csr',dtype=np.float32)

X_train_interaction = interaction_transform(X_train, train_scaled, train_counts, q25,q75)
assert X_train_interaction.shape==(6201,13153)
print('Train q25/q75:',q25,q75,'Features:',X_train_interaction.shape[1])

Train q25/q75: 7.0 27.0 Features: 13153


## 3. 최종 모델 학습 — 6,201명 전체 Train

**주의:** 아래 학습 점수는 Train 자체 점수이므로 일반화 성능이 아닙니다. 최종 후보 선정은 실험 09의 `final_selection_cv.csv` 기준으로 이미 수행했습니다.

In [4]:
# 실험09의 SGD_B_Interaction 설정을 그대로 유지
model = SGDClassifier(loss='log_loss',alpha=0.001,class_weight='balanced',
                      max_iter=1500,tol=1e-3,random_state=SEED)
model.fit(X_train_interaction,y)
train_pred=model.predict(X_train_interaction)
print('학습 성공 | Train 자체 Macro F1 (참고용, 일반화 점수 아님):',
      round(f1_score(y,train_pred,labels=np.arange(26),average='macro',zero_division=0),4))

학습 성공 | Train 자체 Macro F1 (참고용, 일반화 점수 아님): 0.8364


## 4. Test 변환 및 예측 전용 단계

이 셀에서 **처음으로** Test 내용을 읽습니다. Test의 평균/최빈값/분포로 전처리하거나 인코더를 재학습하지 않습니다.

In [5]:
# schema는 Train에서 정해진 gene_columns와 set 일치 검증
schema=pd.read_csv(test_path,nrows=0)
assert 'SUBCLASS' not in schema.columns
assert 'ID' in schema.columns
assert set(schema.columns)-{'ID'}==set(gene_columns), 'Train/Test feature schema 불일치'

Xtest_chunks=[]; test_ids=[]; total_missing=0
for chunk in pd.read_csv(test_path,dtype='string',chunksize=300):
    test_ids.extend(chunk['ID'].astype(str).tolist())
    # 오직 고정 규칙으로 변환; 결측 개수는 감사 로그만 기록
    total_missing += int(chunk[gene_columns].isna().sum().sum())
    Xtest_chunks.append(as_binary(chunk,gene_columns))
X_test=sparse.vstack(Xtest_chunks,format='csr').astype(np.float32)
test_ids=np.asarray(test_ids)
assert X_test.shape==(2546,len(gene_columns)) and len(set(test_ids))==len(test_ids)
test_counts=np.diff(X_test.indptr).astype(np.float32)
test_scaled=count_scaler.transform(np.log1p(test_counts).reshape(-1,1)).astype(np.float32)
X_test_interaction=interaction_transform(X_test,test_scaled,test_counts,q25,q75)
assert X_test_interaction.shape[1]==X_train_interaction.shape[1]
pred_id=model.predict(X_test_interaction)
predicted=enc.inverse_transform(pred_id)
assert set(predicted).issubset(set(classes))
print('Test rows:',len(predicted),'Test missing feature cells (audit only):',total_missing)

Test rows: 2546 Test missing feature cells (audit only): 237


## 5. 샘플 제출 형식 확인 및 submission.csv 생성

In [6]:
sample=pd.read_csv(sample_path,dtype={'ID':'string','SUBCLASS':'string'})
assert list(sample.columns)==['ID','SUBCLASS'],f'샘플 제출 열 확인 필요: {sample.columns.tolist()}'
assert len(sample)==len(predicted)
assert sample['ID'].notna().all() and sample['ID'].is_unique
assert set(sample['ID'].astype(str))==set(test_ids)
# sample ID 순서에 정확히 맞춰 재정렬
prediction_map=dict(zip(test_ids,predicted))
submission=sample[['ID']].copy()
submission['SUBCLASS']=submission['ID'].astype(str).map(prediction_map)
assert submission['SUBCLASS'].notna().all()
assert submission['SUBCLASS'].isin(classes).all()
assert submission.shape==(2546,2)
submission.to_csv(OUT/'submission.csv',index=False,encoding='utf-8-sig')
print('생성:',(OUT/'submission.csv').resolve())
print(submission.head(5).to_string(index=False))

생성: /mnt/data/experiment_10_final/experiment_10/submission.csv
       ID SUBCLASS
TEST_0000     LUSC
TEST_0001     STES
TEST_0002     THCA
TEST_0003      LGG
TEST_0004     LUSC


## 6. 모델·전처리 기준 및 실행 기록 저장

In [7]:
# 재현성: 모델에 필요한 모든 train-fit 객체를 저장
bundle={'model':model,'encoder':enc,'count_scaler':count_scaler,
        'gene_columns':gene_columns,'q25':float(q25),'q75':float(q75),
        'missing_rule':'WT=0, observed non-WT=1, missing=0 (unknown is NOT WT)',
        'model_choice':'Experiment 09 final_selection_cv.csv: SGD_B_Interaction',
        'train_rows':int(X_train.shape[0]),'seed':SEED}
joblib.dump(bundle,OUT/'final_model.joblib',compress=3)
metadata={
 'project':'Cancer type classification / Experiment 10',
 'candidate':'SGD_B_Interaction','algorithm':'SGDClassifier',
 'loss':'log_loss','alpha':0.001,'class_weight':'balanced','random_state':SEED,
 'train_rows':int(X_train.shape[0]),'test_rows':int(X_test.shape[0]),
 'genes':len(gene_columns),'expanded_features':int(X_train_interaction.shape[1]),
 'classes':classes.tolist(),'train_mutation_q25':float(q25),'train_mutation_q75':float(q75),
 'test_missing_cells_audit_only':total_missing,
 'test_used_for_training_or_tuning':False,
 'selection_source':'Experiment 09 Train-only final_selection_cv: mean macro F1 0.384263',
 'validation_note':'0.384263 is candidate-selection CV, NOT public Test score or independent holdout performance',
 'missing_handling':'Fixed encoding: unknown/missing -> 0 for inference; biologically not confirmed WT',
 'train_source_name':train_path.name,'test_source_name':test_path.name,
 'model_bundle':'final_model.joblib','submission_file':'submission.csv'
}
(OUT/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved model, submission, metadata to:', OUT.resolve())

Saved model, submission, metadata to: /mnt/data/experiment_10_final/experiment_10


## 7. 저장된 모델로 다시 예측하는 방법 (선택)

이미 저장한 `final_model.joblib`은 학습 데이터 없이도 동일한 유전자 열 순서와 전처리 규칙을 재적용할 수 있도록 모델과 스케일러, 경계를 함께 포함합니다. 제출 시에는 `experiment_10/submission.csv`만 제출하면 됩니다.

**연구 해석:** 실험 08에서 Interaction이 개선됐고 실험 09의 최종 후보 선택 CV에서 SGD_B_Interaction이 최고였으므로 이 설정을 채택했습니다. 이것이 새로운 임상 환자에 대한 진단 성능을 보장하는 것은 아닙니다.